In [1]:
import sys

sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/utils")
sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/database")

In [2]:
# Base packages -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import logging
import re
import ast
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook

# Paraellel Processing -----------------------------------------------------------------------
import dask
from dask import distributed, dataframe as dd
from dask.diagnostics import ProgressBar
from dask.distributed import  wait, as_completed

# Model work --------------------------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer
import torch


In [3]:
# hide warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Disable TensorFlow INFO and WARNING messages
import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Disable TensorFlow ERROR messages (optional)

import warnings
# Filter out Dask client warnings
warnings.filterwarnings("ignore")


In [4]:
import chromaDB as cbd # specter_ef, create_text_splitter, create_chroma_client
from db_tools import db_connection
from utils import import_config

config, keys = import_config()

# Chroma setup --------------------------------------------------------------------------------
chroma_server_host = "3.88.178.230"

chroma_client = cbd.create_chroma_client(chroma_server_host)


Nanosecond heartbeat on server 1690947950280533512000
[Collection(name=langchain_store), Collection(name=abstracts), Collection(name=fulltext), Collection(name=specter_abstracts), Collection(name=fulltext_specter)]


In [5]:
# Modelish stuff --------------------------------------------------------------------------------
model = AutoModel.from_pretrained("allenai/specter")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")

# Create embeddings function with specter model
specter_embeder = cbd.specter_ef(model, tokenizer)

specter_embeder.create_text_splitter()

In [6]:
# Working Collection
collection = chroma_client.get_or_create_collection("specter_abstracts", embedding_function=specter_embeder)

collection.count()

74595

In [9]:
abstracts = pd.read_csv("/home/ubuntu/work/data/abstracts.csv")

abstracts = abstracts.drop(columns = ['Unnamed: 0', 'id'])

N existing ids:  2050
(454618, 2)
(452568, 2)


Functionalized

In [11]:
# @dask.delayed
def create_split_text(df): 
    
    df['documents'] = df.apply(lambda x: specter_embeder.split_text(x['documents']), axis=1)
    
    return df

def create_ids(df): 
    """ Create ids for each chunk """
    
    df['ids'] = df.apply(lambda x: [f"{x['corpusId']}-{i}" for i in range(len(ast.literal_eval(x['documents'])))], axis=1)
    
    return df

def create_metadatas(df): 
    """ Create metadatas for each chunk """
    
    df['metadatas'] = df.apply(lambda x: [{'corpusid': x['corpusId'], 'chunk':i} for i in range(len(ast.literal_eval(x['documents'])))], axis=1)
    
    return df

def create_embeddings(df): 
    
    df['embeddings'] = df.apply(lambda x: specter_embeder.embed_documents(x['documents']), axis=1)
    
    return df

def create_dict_for_upload(df): 
    
    df = df.apply(lambda x: x.apply(ast.literal_eval))
    
    dict_series = df.to_dict('records')
    
    return dict_series

# Dask functions

# Add documents to collection

- map ddf partitions
- apply function over 
- convert row to dictionary
- process with `process_dict`

In [12]:
# log errors to file
logging.basicConfig(filename='/home/ubuntu/work/therapeutic_accelerator/db_work/chroma_db/db_chroma_setup/abstract_embeddings_dask.log', level=logging.ERROR)

In [30]:
# ab_tester = abstracts.iloc[:10000, :]
n_partitions = len(abstracts) // 100

print(n_partitions)

abstracts_split = np.array_split(abstracts, n_partitions)

4525


In [31]:
def remove_exisiting(df, collection):
    """ Remove existing ids from dataframe """
    # get unique ids from collection that are the corpus ids
    query_filter = {
        "$or": [{'corpusid': str(c)} for c in df['corpusId'].unique()]
    }

    # get ids in df from collection
    temp = collection.get(
        include=["metadatas"],
        where=query_filter
    )
    
    test_ids = pd.DataFrame(temp['metadatas'])
    
    existing_ids = test_ids.astype(str)['corpusid'].unique()
    
    df = df[~df['corpusId'].astype(str).isin(existing_ids)]
    
    return df

In [32]:
for i in tqdm(range(len(abstracts_split))):
    abstracts_split[i] = remove_exisiting(abstracts_split[i], collection)
    if len(abstracts_split[i]) > 0: 
        df = create_split_text(abstracts_split[i])
        df = create_embeddings(df)
        df = create_ids(df)
        df = create_metadatas(df)
        df = results.drop(columns = ['corpusId'])
        dict_series = results.to_dict('records')
        
        temp_dict = {
            'documents': [], 
            'ids': [], 
            'embeddings': [], 
            'metadatas': []
        }

        # combine dictionaries into one
        for ds in dict_series:
            for k in temp_dict.keys(): 
                temp_dict[k] = temp_dict[k] + ds[k]
                
        try: 
            collection.add(**temp_dict)
            print(collection.count())
        except Exception as e: 
            logging.exception("Error occurred while adding documents to collection")
    
        


  2%|▏         | 83/4525 [02:57<2:38:35,  2.14s/it]


NameError: name 'results' is not defined